# 00 — Start here: question, contract, and offline workspace

**Estimated time:** 20 minutes<br>
**Prerequisites:** none<br>
**Learner-produced evidence:** a ready/not-ready asset table and a written learning goal

## Learning objectives

- Understand the experiment question and structured-output task.
- Verify that the local study assets are present without using a network.
- Distinguish a learning result from a production-readiness claim.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## How to use this course

A **Markdown cell** explains an idea; a **code cell** performs a
small local experiment. Read the explanation first, select the
`AAI Local Fine-Tuning (offline)` kernel, and press **Shift+Enter**
to run one cell at a time. Cells headed **Setup — run, do not edit**
are plumbing rather than lesson exercises.

When a cell returns a table or dictionary, interpret it in three
steps:

1. **What does it say?** Describe the measured value without judgment.
2. **What would concern me?** Connect the value to a failure risk.
3. **What would I do next?** Name a check, mitigation, or stop condition.

The notebooks contain worked examples, then exercises and checkpoints.
Generated `.ipynb` files are outputs: maintainers change the narrative
in `scripts/render_notebooks.py` or `scripts/notebook_pedagogy.py` and
regenerate them.


## Why this matters

Fine-tuning can make a model look better in a demo while making the overall system less reliable. Before touching data or weights, you need a precise question, a comparison point, and a definition of acceptable behavior. This notebook establishes those boundaries and verifies that the plane-ready workspace actually contains the evidence-producing assets.

## Key terms in plain language

- **large language model (LLM):** a model trained to predict tokens; useful language behavior emerges from that prediction task, but its output is not automatically a fact or a policy decision.
- **base model:** the unchanged model checkpoint on which prompting or an adapter is built.
- **inference:** running a trained model to produce an output; no weights are learned.
- **fine-tuning:** continuing training on task-specific examples so some learned parameters change.
- **LoRA adapter:** a small set of learned low-rank updates used with a frozen base model; it is not a complete model by itself.
- **structured output:** an answer that must obey a machine-checkable contract such as known JSON fields and allowed labels.
- **baseline:** a reproducible existing approach that a proposed change must beat.
- **train / validation / test:** examples used respectively to learn, choose among changes, and estimate the final chosen system's behavior on untouched data.


## Mental model — how to think about this

Treat model development as a controlled scientific comparison, not a talent show. Write the question, measure a baseline, make one named change, evaluate both under the same contract, and then decide. The lifecycle is **baseline → change → result → decision**. A successful training command proves only that training ran; it does not prove that the change helped.

### Running example

The input `"I forgot my password"` belongs to the known intent `recover_password`. A usable answer is not free-form prose; it is a validated object such as `{"intent": "recover_password", "category": "account", "requires_escalation": false, "response": "I can help you reset your password."}`. A right intent in broken JSON still fails the system contract.

### Questions to ask before continuing

- What user or system decision will this model output support?
- Which behavior must improve, and which safety or format properties may not regress?
- What unchanged baseline and untouched examples make the comparison credible?
- What artifact would let another person reproduce or challenge the conclusion?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **State the contract before implementation.** Name the input, allowed output, quality metrics, safety constraints, and promotion thresholds up front.
- **Change one explainable thing at a time.** Prompt changes, data changes, base-model changes, and adapter training are separate experimental changes.
- **Measure layers separately.** Classification quality, JSON validity, response-policy compliance, latency, and memory answer different questions and should not be collapsed.
- **Keep lineage.** Record exact data, model, code, configuration, and evaluation fingerprints so that a score has meaning later.
- **Start with the smallest useful experiment.** A smoke run catches broken plumbing; a full run is justified only after the evidence path works end to end.

## Common mistakes and why they fail

- **Equating lower training loss with product improvement.** Loss measures the training objective, not generalization, output validity, safety, or user value.
- **Judging a few attractive examples by eye.** Hand-picked outputs hide base rates and failure slices; use a fixed evaluation set and bounded error review.
- **Using the test set while designing.** That turns the test into development data and makes the reported result optimistic.
- **Assuming a public dataset is production-ready.** Availability says nothing by itself about license scope, consent, representativeness, or fitness for the intended use.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Risk guidance:** [NIST AI RMF Generative AI Profile](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)
- **Tool guidance:** [MLflow evaluation-driven development overview](https://mlflow.org/docs/latest/genai/eval-monitor)


## The experiment question

Can a small LoRA adapter improve structured customer-support intent
prediction over the strongest meaningful baseline while preserving
strict JSON, known labels, and safe response wording?

The evidence lifecycle is **baseline → change → result → decision**.
Training success alone is not a decision.


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Check this prepared machine

The paths below stay local. A row with `ready: false` means preparation
is incomplete; it does not trigger a download. The cell stops rather
than letting a missing asset become an in-flight surprise. The model,
source archive, generated data, adapters, and MLflow database are
deliberately ignored by Git.


In [ ]:
from aai_local_finetuning.offline import (
    apple_silicon_status,
    asset_checks,
    deny_network,
    prove_socket_denial,
)
from aai_local_finetuning.settings import load_settings

settings = load_settings()
machine = apple_silicon_status()
readiness = [
    {
        "asset": check.name,
        "ready": check.ready,
        "detail": check.detail,
    }
    for check in [machine, *asset_checks(settings)]
]
not_ready = [item for item in readiness if not item["ready"]]
if not_ready:
    failed = ", ".join(item["asset"] for item in not_ready)
    raise RuntimeError(
        "Offline study is not ready. Re-run `make prepare-flight` while "
        f"online, then rehearse with Wi-Fi off. Failed checks: {failed}"
    )
readiness

## Prove the Python guard

This scoped check deliberately denies Python socket connections. The
notebook also enables the supported offline flags before importing
model or tracking libraries. Turning Wi-Fi off once before departure
remains the strongest rehearsal because native libraries are outside
Python's complete control.


In [ ]:
with deny_network():
    prove_socket_denial()
"Python socket guard passed"

## The output boundary

The model must return exactly four typed fields. A plausible-looking
sentence is not enough, and a high intent score cannot conceal invalid
or policy-breaking generated output. In this course, “response-policy
compliant” means only that the output passed a small versioned set of
wording checks; it is not a broad production-safety claim.

A valid example has all four fields and the correct types:

```json
{"intent":"recover_password","category":"account","requires_escalation":false,"response":"I can help you reset your password."}
```

`{"intent":"recover_password"}` is valid JSON but fails the schema
because fields are missing. `intent=recover_password` is not JSON at all.


In [ ]:
from aai_local_finetuning.evaluation import SupportOutput

SupportOutput.model_json_schema()

## Exercise — write your evidence question

Replace the default sentence with the question you want the final
decision to answer. Success means it names a baseline, a change, a
frozen evaluation boundary, and at least one output-quality gate.


In [ ]:
my_evidence_question = (
    "Does the LoRA change beat the strongest baseline on frozen macro-F1 "
    "while meeting strict schema and response-policy gates?"
)
assert all(
    term in my_evidence_question.lower()
    for term in ("lora", "baseline", "frozen", "schema")
)
my_evidence_question

**Hint:** describe the comparison and its evidence, not the result you
hope to see. Keep production suitability outside this learning claim.


## Checkpoint

You can now explain why offline readiness, strict output validation,
and a frozen comparison are three different concerns.

**Next:** `01_dataset_provenance_and_license.ipynb` examines whether the
source may be used for this curriculum and what remains unproven.
